# 訓練段階の防御（Training Stage Defense）

訓練段階では、目的関数・最適化アルゴリズム・学習パラダイムを含む訓練プロセス全体を制御できる。この段階の防御は (A) 敵対的サンプルに対する防御（敵対的訓練）と (B) バックドア攻撃に対する防御に大別される。

## 敵対的訓練（Adversarial Training）

### 基本的な定式化

敵対的訓練は、クリーンデータでの損失と敵対的サンプルでの損失を同時に最小化するmin-max最適化問題として定式化される。

$$
\min_{\boldsymbol{\theta}} \mathbb{E}_{(\mathbf{x}, y) \in \mathcal{D}} \left[ \mathcal{L}_1(f_{\boldsymbol{\theta}}(\mathbf{x}), y) + \lambda \max_{\delta \in \mathcal{S}} \mathcal{L}_2(f_{\boldsymbol{\theta}}(\mathbf{x} + \delta), y) \right]
$$

- $\mathcal{L}_1$: クリーンデータ上の損失（標準精度の保証）
- $\mathcal{L}_2$: 敵対的ロバスト性の損失
- $\lambda$: ロバスト性と精度のトレードオフを調整するパラメータ
- $\mathcal{S}$: 摂動の制約集合（例: $\{\delta: \|\delta\|_\infty \leq \epsilon\}$）

内側の最大化は**攻撃**（最も損失が大きくなる摂動を見つける）、外側の最小化は**防御**（その攻撃に対してもロバストなパラメータを学習する）に対応する。

### ロバスト汎化の向上（Enhancing Robust Generalization）

#### データ拡張ベース

| 手法 | アプローチ |
|------|----------|
| **AVmixup** | 線形補間による軟ラベル付けデータ拡張 |
| **Cropshift** | 多様性と難度バランスを改善する拡張スキーム |
| **DAJAT** | 単純・複雑な拡張の組み合わせ＋分離バッチ正規化層 |

#### 訓練定式化の改善

| 手法 | アプローチ |
|------|----------|
| **TRADES** | 精度とロバスト性のトレードオフを目的関数に明示的に組み込む |
| **MART** | マージン最大化によるロバスト訓練 |
| **複数摂動型** | 異なる摂動タイプ（$L_\infty$, $L_2$等）を訓練に同時に組み込む |

#### 損失ランドスケープベース

損失関数の平坦性がロバスト汎化に関係することを利用する。

| 手法 | アプローチ |
|------|----------|
| **AWP（Adversarial Weight Perturbation）** | 重みへの敵対的摂動を導入し損失ランドスケープの平坦性を正則化 |
| **Robust Weight Perturbation** | ロバストな重み摂動 |
| **Random Weight Perturbation** | ランダムな重み摂動で汎化ギャップを削減 |

#### サンプル重要度ベース

すべてのサンプルを等しく扱うのではなく、重要なサンプルに注力する。

| 手法 | アプローチ |
|------|----------|
| **Probability Margins** | 決定境界への近接性を3種類のマージンで測定し再重み付け |
| **InfoAT** | 高い相互情報量を持つサンプルに集中 |
| **SOVR** | 小さいロジットマージンを持つ重要サンプルにone-vs-rest損失を適用 |

### 攻撃戦略の改善

内側の最大化（攻撃）をより効果的にすることで、防御の質を向上させる。

| 手法 | アプローチ |
|------|----------|
| **Prior-guided FGSM** | 初期化戦略の改善 |
| **Curriculum AT** | 攻撃強度を段階的に増加させるカリキュラム学習 |
| **FAT（Friendly AT）** | 誤分類される敵対的サンプルの損失最小化に焦点 |
| **LAS-AT** | 学習可能な攻撃戦略の導入 |

### 訓練効率の向上

敵対的訓練は通常の訓練よりも計算コストが高い。この問題に対するアプローチ：

| カテゴリ | 手法 | アプローチ |
|---------|------|----------|
| 攻撃ステップ削減 | **FGSM-RS** | ランダム初期化付き1ステップ攻撃 |
| | **Amata** | 攻撃ステップ数を動的に変更 |
| 計算コスト削減 | **Free AT** | 勾配情報を再利用して生成コストを削減 |
| | **YOPO** | 最初の層に集中して高速化 |
| サンプル選択 | **Coreset Selection** | 代表サンプルの選択で計算量削減 |

## バックドア攻撃に対する防御

### 安全な集中訓練（Secure Centralized Training）

毒入れデータが混入した訓練データからクリーンなモデルを学習する。

| 手法 | アプローチ |
|------|----------|
| **ABL** | 早期の訓練エポックで低損失のサンプルを毒入れとして特定し、その後の訓練で学習を解除 |
| **DBD** | 3段階：(1) ラベルなし自己教師あり学習 → (2) 凍結バックボーンで分類器訓練＋高損失サンプルを毒入れとして特定 → (3) 毒入れラベルを除去してファインチューニング |
| **D-ST** | セミ教師ありコントラスト学習で毒入れラベルを除去後、MCE損失で訓練 |
| **D-BR** | 毒入れの学習解除とクリーン学習を交互に実施 |
| **ASD** | 損失ガイド分割とメタ学習で動的にクリーン/毒入れデータを分離 |
| **CBD** | 因果推論の考え方を活用。バックドアモデル $f_B$ の埋め込みから独立にクリーンモデル $f_C$ を訓練 |

### 安全な分散訓練（Secure Decentralized Training）

連合学習（Federated Learning）において、悪意あるクライアントによるバックドア攻撃を防ぐ。

グローバルモデルの更新は以下のように定式化される：

$$
\Delta\boldsymbol{\theta} = \sum_{i=1}^{n} c_i \eta_i \odot \tau(\Delta\boldsymbol{\theta}_i)
$$

- $c_i$: 正規化重み
- $\eta_i$: 学習率
- $\Delta\boldsymbol{\theta}_i$: クライアント $i$ のモデル更新
- $\tau(\cdot)$: 更新に対する操作関数（防御者が制御）

#### 悪意あるクライアントの検出

| 手法 | アプローチ |
|------|----------|
| **Sniper** | グラフ構造でモデル間の類似性を表現し、最大クリークから良好なモデルを集約 |
| **FLAME** | HDBSCANで角偏差の高い更新を悪意として識別 |
| **FedCPA** | 重要パラメータの分析で良好な更新との差異を検出 |
| **FedGame** | トリガーと目標クラスをリバースエンジニアリングし真正スコアを計算 |

#### バックドア関連ニューロンの操作

| 手法 | アプローチ |
|------|----------|
| **FLAME（適応版）** | $\ell_2$ノルムに依存する適応的クリッピングとノイズ追加 |
| **RFOut-1d** | 平均から乖離したニューロン単位の更新を置換 |
| **PartFedAvg** | ランダムにパラメータの一部を選択し、残りをゼロ化 |

#### 学習率の調整

| 手法 | アプローチ |
|------|----------|
| **FoolsGold** | 履歴勾配の類似性が高いクライアント（攻撃者）の学習率を下げる |
| **RLR** | 勾配符号の合計が閾値を超えた場合に正の学習率、それ以外は負の学習率を適用 |

## 参考文献

- [Wu et al. (2023). Defenses in Adversarial Machine Learning: A Survey. Section IV.](https://arxiv.org/abs/2312.08890)
- [Madry et al. (2018). Towards Deep Learning Models Resistant to Adversarial Attacks.](https://arxiv.org/abs/1706.06083)
- [Zhang et al. (2019). Theoretically Principled Trade-off between Robustness and Accuracy (TRADES).](https://arxiv.org/abs/1901.08573)
- [Wu et al. (2020). Adversarial Weight Perturbation Helps Robust Generalization.](https://arxiv.org/abs/2004.05884)
- [Shafahi et al. (2019). Adversarial Training for Free!](https://arxiv.org/abs/1904.12843)